In [61]:
# 2. ZADATAK: Z3

# Kontekst (Po uzoru na SEP2 i JUN1): Menadžer tima treba da rasporedi 5 programera (Ana, Boris, Vuk, Darija, Elena) na 3 različita projekta (P_1, P_2, P_3).

# Ograničenja (Aksiomi):
# Svaki programer mora biti dodeljen tačno jednom projektu.
# Na projektu P_1 moraju da rade tačno dve osobe.
# Ana i Boris su seniori i ne smeju da rade na istom projektu.
# Vuk i Darija su nerazdvojni i moraju da rade na istom projektu.
# Elena ne želi da radi na projektu P_3.

# Zahtevi:
# Korišćenjem biblioteke z3-solver, modelovati ovaj problem, proveriti da li postoji rešenje (zadovoljivost) i ispisati raspored ljudi po projektima ako rešenje postoji

In [62]:
# !pip install z3-solver
from z3 import *

In [63]:
Person = DeclareSort('Person')
Project = DeclareSort('Project')

WorkTogether = Function('WorkTogether', Person, Person, Project, BoolSort()) # osoba x i osoba y rade zajedno na projektu z
AssignedProject = Function('AssignedProject', Person, Project, BoolSort()) # osoba x radi samo na projektu y

In [64]:
ana, boris, vuk, darija, elena = Consts('ana boris vuk darija elena', Person)
x, y = Consts('x y', Person)

p_1, p_2, p_3 = Consts('p_1 p_2 p_3', Project)
z, w = Consts('z w', Project)

In [65]:
axioms = [
    Distinct(ana, boris, vuk, darija, elena),
    Distinct(p_1, p_2, p_3),

    # svaka osoba mora da radi barem na jednom projektu
    ForAll([x], Or(AssignedProject(x, p_1), AssignedProject(x, p_2), AssignedProject(x, p_3))),

    # svaka osoba radi na tacno jednom projektu
    ForAll([x, z, w], Implies(And(AssignedProject(x, z), z != w), Not(AssignedProject(x, w)))),

    ForAll([x, y, z], WorkTogether(x, y, z) == And(AssignedProject(x, z), AssignedProject(y, z))),

    Not(AssignedProject(elena, p_3)),

    ForAll([z], Not(WorkTogether(ana, boris, z))),

    Exists([z], WorkTogether(vuk, darija, z)),

    If(AssignedProject(ana, p_1), 1, 0) + If(AssignedProject(boris, p_1), 1, 0) + If(AssignedProject(vuk, p_1), 1, 0) + If(AssignedProject(darija, p_1), 1, 0) + If(AssignedProject(elena, p_1), 1, 0) == 2
]

In [66]:
s = Solver()
s.add(axioms)

if s.check() == sat:
  model = s.model()
  # print(model)

  osobe = [ana, boris, vuk, darija, elena]
  projekti = [p_1, p_2, p_3]
  for o in osobe:
      for p in projekti:
          if is_true(model.eval(AssignedProject(o, p), model_completion = True)):
              print(f"- {o} ide na projekat {p}")
else:
  print('unsat')

- ana ide na projekat p_1
- boris ide na projekat p_2
- vuk ide na projekat p_3
- darija ide na projekat p_3
- elena ide na projekat p_1
